### Test Pydantic AI with LangChain's SQL tools

In [20]:
from dotenv import load_dotenv
import nest_asyncio
import os

load_dotenv('../.env')
neon_conn_string = os.getenv("NEON_DB_URL")
langchain_neon_conn_string = neon_conn_string.replace('postgresql', 'postgresql+psycopg')
nest_asyncio.apply() # for async issues in Jupyter stuff

gpt_model = "gpt-5-mini"

In [2]:
import logfire
logfire.configure()
logfire.instrument_pydantic_ai()

Logfire project URL: ]8;id=787092;https://logfire-us.pydantic.dev/iellis02/blue-horizon\https://logfire-us.pydantic.dev/iellis02/blue-horizon]8;;\

Get the enums

In [5]:
import psycopg

enums = ['availability_status_type', 'room_bed_type', 'room_status_type', 'room_type']
enum_values = {}

with psycopg.connect(neon_conn_string) as neon_conn:
    with neon_conn.cursor() as cur:
        for enum in enums:
            cur.execute(f"SELECT enum_range(NULL::{enum})")
            enum_values[enum] = [val.strip('\'"{}') for val in cur.fetchall()[0][0].split(',')]
        cur.execute("SELECT DISTINCT unnest(basic_amenities) FROM rooms;")
        basic_amenities = [val[0] for val in cur.fetchall()]
        cur.execute("SELECT DISTINCT unnest(additional_amenities) FROM rooms;")
        additional_amenities = [val[0] for val in cur.fetchall()]
        cur.execute("SELECT DISTINCT unnest(view_type) FROM rooms;")
        view_types = [val[0] for val in cur.fetchall()]
        
            
print(f"enums = {enum_values}")
print(f"basic amenities = {basic_amenities}")
print(f"additional amenities = {additional_amenities}")
print(f"view types = {view_types}")

enums = {'availability_status_type': ['Booked', 'Available', 'Maintenance'], 'room_bed_type': ['Queen', 'Double Queen', 'King', 'Double King', 'King + Sofa Bed', 'King + Multiple Sofa Beds'], 'room_status_type': ['Available', 'Occupied', 'Maintenance'], 'room_type': ['Standard', 'Deluxe', 'Suite', 'Presidential Suite']}
basic amenities = ['Full Kitchen', 'Executive Office', 'Nespresso Machine', 'Premium Coffee Maker', 'High-Speed WiFi', 'Premium Bathrobes', 'Kitchenette', 'Bathrobes', "Butler's Pantry", 'Welcome Amenity', 'Full-Size Refrigerator', '55" Smart TV', 'Bluetooth Speaker', 'Professional Coffee Bar', 'Guest Bathroom', 'Air Conditioning', 'Living Room', 'Luxury Welcome Amenity', 'Bang & Olufsen Sound System', 'Ultra-High-Speed WiFi', 'Work Desk', 'Multiple 75" Smart TVs', 'In-Room Safe', 'Smart TV', 'Living Room Area', 'Personalized Stationery', 'Hair Dryer', 'Walk-in Closet', 'Multiple Bathrooms', 'Dining Area', 'Bose Sound System', '65" Smart TV', 'Slippers', 'Evening Turndo

In [26]:
from langchain_community.utilities import SQLDatabase

schema_description = {
    "rooms": (f"""
            CREATE TABLE rooms (
                room_id INT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY, -- Do not return
                room_number INT NOT NULL,
                floor INT NOT NULL,
                type room_type, -- Enum with options {enum_values['room_type']}
                square_feet INT,
                basic_amenities TEXT[],  -- Options are {basic_amenities}
                additional_amenities TEXT[], -- Options are {additional_amenities}
                max_occupancy INT,
                bed_type room_bed_type,  -- Enum with options {enum_values['room_bed_type']}
                view_type TEXT[],  -- Options are {view_types}
                accessibility BOOLEAN,  -- Whether handicapped accessible
                status room_status_type, -- Enum with options {enum_values['room_status_type']}, do not return
                last_renovation DATE, -- Do not provide unless asked for
                base_rate NUMERIC(10, 2),  -- Do not provide unless asked for
                max_rate NUMERIC(10, 2)  -- Do not provide unless asked for
                -- The table lists details about all the rooms in the hotel.
            );
    """),
    "room_availability": (f"""
            CREATE TABLE room_availability (
                id INT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY, -- Do not return
                room_id INT NOT NULL, -- Do not return
                room_number INT NOT NULL,
                date DATE NOT NULL,
                status availability_status_type,  -- Enum with options {enum_values['availability_status_type']}
                price NUMERIC(8,2),
                max_occupancy INT,
                FOREIGN KEY (room_id) REFERENCES rooms(room_id)
                -- The table lists the room availability by date and the corresponding rate
            );
    """),
}

db = SQLDatabase.from_uri(database_uri=langchain_neon_conn_string, include_tables=schema_description.keys(), custom_table_info=schema_description)

In [27]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain_openai import ChatOpenAI
from pydantic_ai.ext.langchain import LangChainToolset

# 1. Initialize your Language Model (LLM)
# Ensure your LLM supports Pydantic AI/Function calling (e.g., OpenAI, Gemini)
llm = ChatOpenAI(model=gpt_model, temperature=0)

# 2. Initialize the LangChain SQL Toolkit
# This toolkit will use the 'db' object with your custom schema info.
toolkit = SQLDatabaseToolkit(db=db, llm=llm)

# 3. Wrap the LangChain Toolkit into a Pydantic AI Toolset
sql_toolset = LangChainToolset(toolkit.get_tools())

In [35]:
from langchain_classic import hub

prompt_template = hub.pull("langchain-ai/sql-agent-system-prompt")
system_prompt = prompt_template.format(dialect="PostgreSQL", top_k=3)
system_prompt += "\nIf you receive an empty result from the SQL query, simply state that you couldn't find anything."

In [36]:
from pydantic_ai import Agent

# Create the Pydantic AI Agent
sql_agent = Agent(
    'openai:' + gpt_model,
    toolsets=[sql_toolset],
    system_prompt=system_prompt,
)

In [30]:
prompt = 'Show me rooms with a view of the ocean and a 55" TV'
result = await sql_agent.run(prompt)

16:49:37.302 sql_agent run
16:49:37.305   chat gpt-5-mini
16:49:39.930   running 1 tool
16:49:39.931     running tool: sql_db_list_tables
16:49:39.936   chat gpt-5-mini
16:49:42.129   running 1 tool
16:49:42.129     running tool: sql_db_schema
16:49:42.133   chat gpt-5-mini
16:49:49.220   running 1 tool
16:49:49.220     running tool: sql_db_query_checker
16:50:05.662   chat gpt-5-mini
16:50:09.045   running 1 tool
16:50:09.047     running tool: sql_db_query
16:50:09.134   chat gpt-5-mini


In [31]:
print(result.output)

I found these rooms with an ocean view and a 55" Smart TV (limited to 3 results):

- Room 502 — Floor 5 — Deluxe (Double Queen) — Max occupancy: 3  
  Views: Pool View, Ocean View  
  Amenities (selected): 55" Smart TV, Air Conditioning, Nespresso Machine, Mini Fridge, High-Speed WiFi, In-Room Safe

- Room 503 — Floor 5 — Deluxe (Double Queen) — Max occupancy: 3  
  Views: Pool View, Ocean View  
  Amenities (selected): 55" Smart TV, Air Conditioning, Nespresso Machine, Mini Fridge, High-Speed WiFi, In-Room Safe

- Room 504 — Floor 5 — Deluxe (Double Queen) — Max occupancy: 3  
  Views: Ocean View  
  Amenities (selected): 55" Smart TV, Air Conditioning, Nespresso Machine, Mini Fridge, High-Speed WiFi, In-Room Safe

Would you like me to check availability or show more rooms?


In [32]:
prompt = "I'd like a room for May 20, 2025 for less than $500 with a view of the city."
result = await sql_agent.run(prompt)

16:50:37.506 sql_agent run
16:50:37.508   chat gpt-5-mini
16:50:39.119   running 1 tool
16:50:39.120     running tool: sql_db_list_tables
16:50:39.125   chat gpt-5-mini
16:50:40.759   running 1 tool
16:50:40.759     running tool: sql_db_schema
16:50:40.763   chat gpt-5-mini
16:50:49.729   running 1 tool
16:50:49.730     running tool: sql_db_query_checker
16:51:00.702   chat gpt-5-mini
16:51:03.374   running 1 tool
16:51:03.374     running tool: sql_db_query
16:51:03.494   chat gpt-5-mini


In [33]:
print(result.output)

Here are up to 3 rooms available May 20, 2025 under $500 with a city view:

- Room 112 — Floor 1 — Standard — Queen bed — $326.17 — Max occupancy 2 — Views: City View, Courtyard View
- Room 319 — Floor 3 — Standard — Queen bed — $328.46 — Max occupancy 2 — Views: City View, Courtyard View
- Room 220 — Floor 2 — Standard — Double Queen bed — $328.77 — Max occupancy 2 — Views: City View, Courtyard View

Would you like to book one of these, see more options, or filter by bed type or floor?


In [37]:
prompt = 'Does the hotel have any rooms with a cold plunge pool and a view of the Eiffel Tower?'
result = await sql_agent.run(prompt)

16:58:07.952 sql_agent run
16:58:07.954   chat gpt-5-mini
16:58:09.550   running 1 tool
16:58:09.551     running tool: sql_db_list_tables
16:58:09.553   chat gpt-5-mini
16:58:11.907   running 1 tool
16:58:11.908     running tool: sql_db_schema
16:58:11.912   chat gpt-5-mini
16:58:24.438   running 1 tool
16:58:24.439     running tool: sql_db_query_checker
16:58:40.893   chat gpt-5-mini
16:58:44.296   running 1 tool
16:58:44.296     running tool: sql_db_query
16:58:44.384   chat gpt-5-mini
16:58:51.490   running 1 tool
16:58:51.491     running tool: sql_db_query_checker
16:59:04.759   chat gpt-5-mini
16:59:07.410   running 1 tool
16:59:07.411     running tool: sql_db_query
16:59:07.498   chat gpt-5-mini


In [38]:
print(result.output)

I couldn't find any rooms that match both criteria (a cold/plunge pool in additional_amenities and an Eiffel Tower view in view_type).

Would you like me to:
- List rooms that have a plunge/cold pool (regardless of view)?
- List rooms that have an Eiffel Tower view (regardless of amenities)?
- Broaden the search to other private pool or terrace options?
